# Underflow, Overflow, and Stable Softmax

Floating-point numbers have a finite range and finite precision. Operations that exceed the range overflow to infinity, while operations that fall below the smallest representable magnitude underflow to zero. Both behaviors can silently destroy a machine-learning computation.

## Learning goals

- Explain underflow and overflow for IEEE 754 double-precision values.
- Observe Python behavior for very small and very large literals.
- Implement softmax and explain why the naive version fails.
- Prove and apply the shift-invariance trick used by stable softmax.
- Use NumPy to compute softmax over a vector.


## 1. The limits of floating point

Python `float` is a C `double`, normally an IEEE 754 binary64 value with 1 sign bit, 11 exponent bits, and 52 fraction bits. The largest finite value is approximately $1.8 \times 10^{308}$. The smallest positive subnormal value is approximately $4.9 \times 10^{-324}$.

- Values below the subnormal range round to `0.0` (underflow).
- Values above the maximum finite value become `inf` (overflow).
- Some operations raise `OverflowError` instead of returning `inf`; `math.exp` is one example.


In [ ]:
small_value = 1e-100
underflowed_value = 1e-1000
large_value = 1e100
overflowed_value = 1e1000

print("1e-100  ->", small_value)
print("1e-1000 ->", underflowed_value)
print("1e100   ->", large_value)
print("1e1000  ->", overflowed_value)

In [ ]:
import math

try:
    print(math.exp(1000.0))
except OverflowError as error:
    print(type(error).__name__, error)

print("isfinite(inf):", math.isfinite(float("inf")))

## 2. Softmax

Softmax converts a vector of real-valued scores into a probability distribution:

$$
\operatorname{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{n} e^{x_j}}.
$$

Each output is positive and the outputs sum to one. The naive implementation exponentiates the raw scores. If any score is larger than about 709, `math.exp` overflows before the normalization can happen.


In [ ]:
def softmax_naive(values: list[float]) -> list[float]:
    """Return softmax probabilities without numerical protection."""
    exponentials = [math.exp(value) for value in values]
    total = sum(exponentials)
    return [value / total for value in exponentials]


print(softmax_naive([1.0, 2.0, 3.0]))

try:
    print(softmax_naive([1001.0, 1002.0, 1003.0]))
except OverflowError as error:
    print(type(error).__name__, error)

## 3. The stable softmax trick

Let $m = \max_i x_i$. Multiplying the numerator and denominator by $e^{-m}$ gives

$$
\frac{e^{x_i}}{\sum_j e^{x_j}}
=
\frac{e^{x_i} e^{-m}}{\sum_j e^{x_j} e^{-m}}
=
\frac{e^{x_i - m}}{\sum_j e^{x_j - m}}.
$$

The result is mathematically identical, but every exponent is now at most zero, so no term exceeds one and the largest term is exactly $e^0 = 1$. This prevents overflow. Underflow can still occur for extremely small terms, but those terms contribute negligibly to the normalized result.


In [ ]:
def softmax(values: list[float]) -> list[float]:
    """Return a numerically stable softmax distribution."""
    maximum = max(values)
    exponentials = [math.exp(value - maximum) for value in values]
    total = sum(exponentials)
    return [value / total for value in exponentials]


for scores in ([1.0, 2.0, 3.0], [1001.0, 1002.0, 1003.0], [1.0, 1001.0, 1002.0]):
    probabilities = softmax(scores)
    print(probabilities)
    print("sum:", sum(probabilities))

## 4. Vectorized softmax with NumPy

NumPy applies the same shift to every element without a Python loop. The normalized result is a `float64` array whose entries sum to one (up to rounding error).


In [ ]:
import numpy as np


def softmax_numpy(values: np.ndarray) -> np.ndarray:
    """Return a numerically stable softmax for a one-dimensional array."""
    shifted = values - np.max(values)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum()


scores = np.array([1.0, 1001.0, 1002.0])
probabilities = softmax_numpy(scores)

print(probabilities)
print("sum:", probabilities.sum())

## 5. Working in log space

When probabilities themselves underflow, work with log-probabilities and use the log-sum-exp identity:

$$
\log \sum_i e^{x_i} = m + \log \sum_i e^{x_i - m},
\qquad m = \max_i x_i.
$$

`scipy.special.logsumexp` implements this pattern for arrays.


In [ ]:
from scipy.special import logsumexp

log_probabilities = scores - logsumexp(scores)
print(log_probabilities)
print(np.exp(log_probabilities))

## Practice

1. Find the smallest positive subnormal value that Python can represent with `math.ulp(0.0)`.
2. Predict the output of `math.exp(-1000.0)` and verify it.
3. Compute softmax for `[1000.0, 1000.0, 1000.0]` and explain the result.
4. Prove that stable softmax returns the same probabilities as the naive formula.
5. Rewrite `softmax` to raise `ValueError` when the input list is empty.
